In [1]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


### Datasets

In [7]:
X=tf.range(10)
dataset=tf.data.Dataset.from_tensor_slices(X)
dataset

<_TensorSliceDataset element_spec=TensorSpec(shape=(), dtype=tf.int32, name=None)>

In [8]:
for item in dataset:
  print(item)

tf.Tensor(0, shape=(), dtype=int32)
tf.Tensor(1, shape=(), dtype=int32)
tf.Tensor(2, shape=(), dtype=int32)
tf.Tensor(3, shape=(), dtype=int32)
tf.Tensor(4, shape=(), dtype=int32)
tf.Tensor(5, shape=(), dtype=int32)
tf.Tensor(6, shape=(), dtype=int32)
tf.Tensor(7, shape=(), dtype=int32)
tf.Tensor(8, shape=(), dtype=int32)
tf.Tensor(9, shape=(), dtype=int32)


In [9]:
dataset=dataset.repeat(3).batch(7)
for item in dataset:
  print(item)

tf.Tensor([0 1 2 3 4 5 6], shape=(7,), dtype=int32)
tf.Tensor([7 8 9 0 1 2 3], shape=(7,), dtype=int32)
tf.Tensor([4 5 6 7 8 9 0], shape=(7,), dtype=int32)
tf.Tensor([1 2 3 4 5 6 7], shape=(7,), dtype=int32)
tf.Tensor([8 9], shape=(2,), dtype=int32)


In [10]:
dataset=dataset.map(lambda x:x**2)
for item in dataset:
  print(item)

tf.Tensor([ 0  1  4  9 16 25 36], shape=(7,), dtype=int32)
tf.Tensor([49 64 81  0  1  4  9], shape=(7,), dtype=int32)
tf.Tensor([16 25 36 49 64 81  0], shape=(7,), dtype=int32)
tf.Tensor([ 1  4  9 16 25 36 49], shape=(7,), dtype=int32)
tf.Tensor([64 81], shape=(2,), dtype=int32)


In [14]:
dataset=dataset.apply(tf.data.experimental.unbatch())

Instructions for updating:
Use `tf.data.Dataset.unbatch()`.


In [15]:
dataset=dataset.filter(lambda x:x<10)

In [16]:
for item in dataset.take(3):
  print(item)

tf.Tensor(0, shape=(), dtype=int32)
tf.Tensor(1, shape=(), dtype=int32)
tf.Tensor(4, shape=(), dtype=int32)


### Full Example

In [2]:
import sklearn

In [18]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

housing=fetch_california_housing()
X_train_full,X_test,y_train_full,y_test=train_test_split(
  housing.data,housing.target.reshape(-1,1),random_state=42)
X_train,X_valid,y_train,y_valid=train_test_split(
  X_train_full,y_train_full,random_state=42
)

scaler=StandardScaler()
scaler.fit(X_train)
X_mean=scaler.mean_
X_std=scaler.scale_

In [3]:
import os

### Save multiple csv files

In [20]:
def save_to_multiple_csv_files(data, name_prefix, header=None, n_parts=10):
    housing_dir = os.path.join("datasets", "housing")
    os.makedirs(housing_dir, exist_ok=True)
    path_format = os.path.join(housing_dir, "my_{}_{:02d}.csv")

    filepaths = []
    m = len(data)
    for file_idx, row_indices in enumerate(np.array_split(np.arange(m), n_parts)):
        part_csv = path_format.format(name_prefix, file_idx)
        filepaths.append(part_csv)
        with open(part_csv, "wt", encoding="utf-8") as f:
            if header is not None:
                f.write(header)
                f.write("\n")
            for row_idx in row_indices:
                f.write(",".join([repr(col) for col in data[row_idx]]))
                f.write("\n")
    return filepaths

In [21]:
train_data = np.c_[X_train, y_train]
valid_data = np.c_[X_valid, y_valid]
test_data = np.c_[X_test, y_test]
header_cols = housing.feature_names + ["MedianHouseValue"]
header = ",".join(header_cols)

train_filepaths = save_to_multiple_csv_files(train_data, "train", header, n_parts=20)
valid_filepaths = save_to_multiple_csv_files(valid_data, "valid", header, n_parts=10)
test_filepaths = save_to_multiple_csv_files(test_data, "test", header, n_parts=10)

In [23]:
import pandas as pd

pd.read_csv(train_filepaths[0]).head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedianHouseValue
0,np.float64(3.5214),np.float64(15.0),np.float64(3.0499445061043287),np.float64(1.106548279689234),np.float64(1447.0),np.float64(1.6059933407325193),np.float64(37.63),np.float64(-122.43),np.float64(1.442)
1,np.float64(5.3275),np.float64(5.0),np.float64(6.490059642147117),np.float64(0.9910536779324056),np.float64(3464.0),np.float64(3.4433399602385686),np.float64(33.69),np.float64(-117.39),np.float64(1.687)
2,np.float64(3.1),np.float64(29.0),np.float64(7.5423728813559325),np.float64(1.5915254237288134),np.float64(1328.0),np.float64(2.2508474576271187),np.float64(38.44),np.float64(-122.98),np.float64(1.621)
3,np.float64(7.1736),np.float64(12.0),np.float64(6.289002557544757),np.float64(0.9974424552429667),np.float64(1054.0),np.float64(2.6956521739130435),np.float64(33.55),np.float64(-117.7),np.float64(2.621)
4,np.float64(2.0549),np.float64(13.0),np.float64(5.312457454050374),np.float64(1.0850918992511913),np.float64(3297.0),np.float64(2.2443839346494214),np.float64(33.93),np.float64(-116.93),np.float64(0.956)


In [24]:
train_filepaths

['datasets\\housing\\my_train_00.csv',
 'datasets\\housing\\my_train_01.csv',
 'datasets\\housing\\my_train_02.csv',
 'datasets\\housing\\my_train_03.csv',
 'datasets\\housing\\my_train_04.csv',
 'datasets\\housing\\my_train_05.csv',
 'datasets\\housing\\my_train_06.csv',
 'datasets\\housing\\my_train_07.csv',
 'datasets\\housing\\my_train_08.csv',
 'datasets\\housing\\my_train_09.csv',
 'datasets\\housing\\my_train_10.csv',
 'datasets\\housing\\my_train_11.csv',
 'datasets\\housing\\my_train_12.csv',
 'datasets\\housing\\my_train_13.csv',
 'datasets\\housing\\my_train_14.csv',
 'datasets\\housing\\my_train_15.csv',
 'datasets\\housing\\my_train_16.csv',
 'datasets\\housing\\my_train_17.csv',
 'datasets\\housing\\my_train_18.csv',
 'datasets\\housing\\my_train_19.csv']

### Building an Input Pipeline

In [25]:
filepath_dataset=tf.data.Dataset.list_files(train_filepaths,seed=42)

In [26]:
for filepath in filepath_dataset:
  print(filepath)

tf.Tensor(b'datasets\\housing\\my_train_05.csv', shape=(), dtype=string)
tf.Tensor(b'datasets\\housing\\my_train_16.csv', shape=(), dtype=string)
tf.Tensor(b'datasets\\housing\\my_train_01.csv', shape=(), dtype=string)
tf.Tensor(b'datasets\\housing\\my_train_17.csv', shape=(), dtype=string)
tf.Tensor(b'datasets\\housing\\my_train_00.csv', shape=(), dtype=string)
tf.Tensor(b'datasets\\housing\\my_train_14.csv', shape=(), dtype=string)
tf.Tensor(b'datasets\\housing\\my_train_10.csv', shape=(), dtype=string)
tf.Tensor(b'datasets\\housing\\my_train_02.csv', shape=(), dtype=string)
tf.Tensor(b'datasets\\housing\\my_train_12.csv', shape=(), dtype=string)
tf.Tensor(b'datasets\\housing\\my_train_19.csv', shape=(), dtype=string)
tf.Tensor(b'datasets\\housing\\my_train_07.csv', shape=(), dtype=string)
tf.Tensor(b'datasets\\housing\\my_train_09.csv', shape=(), dtype=string)
tf.Tensor(b'datasets\\housing\\my_train_13.csv', shape=(), dtype=string)
tf.Tensor(b'datasets\\housing\\my_train_15.csv', sh

In [27]:
n_readers=5
dataset=filepath_dataset.interleave(
  lambda filepath: tf.data.TextLineDataset(filepath).skip(1),
  cycle_length=n_readers
)

In [28]:
for line in dataset.take(5):
  print(line.numpy())

b'np.float64(4.5909),np.float64(16.0),np.float64(5.475877192982456),np.float64(1.0964912280701755),np.float64(1357.0),np.float64(2.9758771929824563),np.float64(33.63),np.float64(-117.71),np.float64(2.418)'
b'np.float64(2.4792),np.float64(24.0),np.float64(3.4547038327526134),np.float64(1.1341463414634145),np.float64(2251.0),np.float64(3.921602787456446),np.float64(34.18),np.float64(-118.38),np.float64(2.0)'
b'np.float64(4.2708),np.float64(45.0),np.float64(5.121387283236994),np.float64(0.953757225433526),np.float64(492.0),np.float64(2.8439306358381504),np.float64(37.48),np.float64(-122.19),np.float64(2.67)'
b'np.float64(2.1856),np.float64(41.0),np.float64(3.7189873417721517),np.float64(1.0658227848101265),np.float64(803.0),np.float64(2.0329113924050635),np.float64(32.76),np.float64(-117.12),np.float64(1.205)'
b'np.float64(4.1812),np.float64(52.0),np.float64(5.701388888888889),np.float64(0.9965277777777778),np.float64(692.0),np.float64(2.4027777777777777),np.float64(33.73),np.float64(-118

## TFRecords

In [4]:
with tf.io.TFRecordWriter("my_data.tfrecord")as f:
  f.write(b"This is the first record")
  f.write(b"This is the second record")


In [5]:
filepaths=["my_data.tfrecord"]
dataset=tf.data.TFRecordDataset(filepaths)
for item in dataset:
  print(item)

tf.Tensor(b'This is the first record', shape=(), dtype=string)
tf.Tensor(b'This is the second record', shape=(), dtype=string)


### Compressed Record

In [6]:
options=tf.io.TFRecordOptions(compression_type="GZIP")
with tf.io.TFRecordWriter("my_compressed.tfrecord",options) as f:
  f.write(b"This is the first record")
  f.write(b"This is the second record")


In [7]:
dataset=tf.data.TFRecordDataset(["my_compressed.tfrecord"],
                                compression_type="GZIP")
for item in dataset:
  print(item)

tf.Tensor(b'This is the first record', shape=(), dtype=string)
tf.Tensor(b'This is the second record', shape=(), dtype=string)


### ProtoBuff

In [8]:
%%writefile person.proto
syntax ="proto3"
message Person {
  string name=1;
  int32 id=2;
  repeated string email=3;
}

Writing person.proto


In [ ]:
person = Person(name="Al", id=123, email=["a@b.com"])  
print(person) 

In [ ]:
# Serlialize to String
s=person.SerlializeToString()
s

In [ ]:
# Deserliaize 
person2.ParseFromString(s)

### Built in ProtoBuf

In [18]:
from tensorflow.train import BytesList,FloatList,Int64List
from tensorflow.train import Example,Features,Feature

person_example=Example(
  features=Features(
    feature={
      "name":Feature(bytes_list=BytesList(value=[b"Alice"])),
      "id":Feature(int64_list=Int64List(value=[123])),
      "emails":Feature(bytes_list=BytesList(value=[b"a@b.com"]))
    }
  )
)

## Preprocessing the Input Features

In [20]:
vocab = ["<1H OCEAN", "INLAND", "NEAR OCEAN", "NEAR BAY", "ISLAND"]

indices = tf.cast(tf.range(len(vocab)), tf.int64) 

table_init = tf.lookup.KeyValueTensorInitializer(vocab, indices)
num_oov_buckets = 2
table = tf.lookup.StaticVocabularyTable(table_init, num_oov_buckets)

test_categories = tf.constant(["NEAR BAY", "DESERT", "INLAND"])
print(table.lookup(test_categories))

tf.Tensor([3 5 1], shape=(3,), dtype=int64)


In [ ]:
regular_inputs = keras.layers.Input(shape=[8])
categories = keras.layers.Input(shape=[], dtype=tf.string)
cat_indices = keras.layers.Lambda(lambda cats: table.lookup(cats))(categories)
cat_embed = keras.layers.Embedding(input_dim=6, output_dim=2)(cat_indices)
encoded_inputs = keras.layers.concatenate([regular_inputs, cat_embed])
outputs = keras.layers.Dense(1)(encoded_inputs)
model = keras.models.Model(inputs=[regular_inputs, categories],
 outputs=[outputs])


## TF Transform

In [10]:
try:
  import tensorflow_transform as tft

  def preprocess(inputs):
    median_age=inputs["housing_median_age"]
    ocean_proximity=inputs["ocean_proximity"]
    standardized_age=tft.scale_to_z_score(median_age-tft.mean(median_age))
    ocean_proximity_id=tft.compute_and_apply_vocabulary(ocean_proximity)
    return{
      "standardized_median_age":standardized_age,
      "ocean_proximity_id":ocean_proximity_id
    }
except ImportError:
  print("TF Transform is not installed. Try running: pip install -U tensorflow-transform")

TF Transform is not installed. Try running: pip install -U tensorflow-transform


## TensorFlow Dataset TFDS

In [11]:
import tensorflow_datasets as tfds

dataset=tfds.load(name='mnist')
mnist_train,mnist_test=dataset["train"],dataset["test"]

c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dl Completed...: 0 url [00:00, ? url/s]
Dl Completed...:  50%|█████     | 2/4 [00:01<00:01,  1.67 url/s]

Dl Completed...: 100%|██████████| 4/4 [00:04<00:00,  1.05 url/s]

Dl Completed...: 100%|██████████| 4/4 [00:04<00:00,  1.03s/ url]
                                                                        

Dataset mnist downloaded and prepared to C:\Users\HP\tensorflow_datasets\mnist\3.0.1. Subsequent calls will reuse this data.
